# Day 3 - Lab 2: Trees, forests and boosting

**Goal:** move from a single decision tree to the ensembles that win on tabular data, read which features actually drive a model, and meet the multiclass confusion matrix.

In Lab 1 the honest classifier hit a ceiling around 69%. Here we ask whether more powerful models can push past it, and learn to be honest when they cannot.

Day 3 adds three modelling packages (xgboost, lightgbm, mlxtend), pinned in `requirements.txt` and installed with the rest of the environment. This cell only confirms they are present; it does not install anything, so it cannot disturb the pinned stack. If a package is missing, run `pip install -r requirements.txt` in your activated `.venv` and restart the kernel.

In [ ]:
import importlib.util
missing = [p for p in ['xgboost', 'lightgbm'] if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(
        f"Missing packages {missing}. In your activated .venv run:  "
        f"pip install -r requirements.txt   then restart the kernel.")
import xgboost, lightgbm
print(f'xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__} ready')

## 1. Load the same table and honest features

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()
df = pd.read_csv(DATA / 'processed' / 'service_requests_features.csv')

honest = ['submitted_hour', 'submitted_dow', 'submitted_month', 'is_weekend',
          'is_digital', 'population', 'priority_rank', 'target_resolution_hours']
lab = df[df['sla_met'].notna()].copy()
lab['sla_met'] = lab['sla_met'].astype(int)

X, y = lab[honest], lab['sla_met']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('train:', Xtr.shape, ' test:', Xte.shape)

## 2. A single decision tree, and why depth is dangerous

A decision tree splits the data into ever-purer groups. Let it grow without limit and it will memorise the training set: near-perfect on data it has seen, much weaker on data it has not. That gap is **overfitting**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# TODO: fit two trees on (Xtr, ytr):
#   - a DEEP tree with no depth limit
#   - a SHALLOW tree with max_depth=4
# For each, print TRAIN and TEST accuracy.
# Expect the deep tree near 0.99 on train but far lower on test (overfitting);
# the shallow tree much closer between the two.
# Keep them in variables called `deep` and `shallow`; later cells use them.


The deep tree's training score races ahead of its test score: it has memorised noise. The shallow tree gives up a little training accuracy for honesty on unseen data. Ensembles are the principled way to get the best of both.

### See it: where overfitting begins

Two trees give two data points. Sweeping depth from 1 to 20 gives the whole story, and this is the single most useful diagram in supervised learning: the point where the two lines part company is the moment the model stops learning the signal and starts memorising the noise.

In [ ]:
import matplotlib.pyplot as plt

depths = range(1, 21)
tr_scores, te_scores = [], []
for dep in depths:
    t = DecisionTreeClassifier(max_depth=dep, random_state=42).fit(Xtr, ytr)
    tr_scores.append(accuracy_score(ytr, t.predict(Xtr)))
    te_scores.append(accuracy_score(yte, t.predict(Xte)))

best = list(depths)[int(np.argmax(te_scores))]
plt.figure(figsize=(9, 5))
plt.plot(depths, tr_scores, 'o-', label='training accuracy', linewidth=2)
plt.plot(depths, te_scores, 's-', label='test accuracy', linewidth=2)
plt.axvline(best, color='red', linestyle='--', label=f'best test depth = {best}')
plt.fill_between(depths, te_scores, tr_scores, alpha=0.15, color='red')
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.title('The overfitting gap: training rises, test does not')
plt.xticks(list(depths)); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The shaded wedge *is* overfitting: every bit of training accuracy bought past the red line is memorisation, not learning. Note the test curve does not collapse so much as plateau, then drift down. That plateau is the real ceiling of this problem.

## 3. Random forest: many trees, decorrelated

A random forest grows hundreds of trees, each on a random slice of the rows and columns, and averages them. The individual trees still overfit, but their *errors* are different, so averaging cancels much of the noise. This is **bagging**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print('random forest  test %.3f' % accuracy_score(yte, rf.predict(Xte)))

## 4. Gradient boosting: the tabular workhorse

Boosting builds trees in sequence, each one correcting the previous one's mistakes. On structured, tabular data like this, gradient boosting (XGBoost, LightGBM) is what practitioners actually reach for. It is the natural step up from a random forest: trees, then bagging, then boosting.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    eval_metric='logloss', random_state=42).fit(Xtr, ytr)
print('xgboost        test %.3f' % accuracy_score(yte, xgb.predict(Xte)))

## 5. Put them side by side

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
scores = {
    'Logistic (Lab 1)': accuracy_score(yte, logreg.predict(Xte)),
    'Deep tree':        accuracy_score(yte, deep.predict(Xte)),
    'Shallow tree':     accuracy_score(yte, shallow.predict(Xte)),
    'Random forest':    accuracy_score(yte, rf.predict(Xte)),
    'XGBoost':          accuracy_score(yte, xgb.predict(Xte)),
}
for name, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f'  {name:20s} {s:.3f}')

Notice what this says. The ensembles comfortably beat the *deep* tree, exactly as the theory promises. But none of them run away from a plain logistic regression or a shallow tree. On a genuinely hard problem, where the signal is weak, model complexity buys little. Reporting that honestly is worth more than quietly picking whichever model looks best on one split.

## 6. Which features actually drive the model?

A forest can report how much each feature contributed to its splits. This is a first read on *why*, not just *how well*.

One caution, and a genuinely useful one to teach: this impurity-based score quietly favours features with many distinct values, because they offer more places to split. Watch whether a high-cardinality feature like the hour of day rises to the top for that reason rather than because it truly matters. Treat importances as a hypothesis to check, not a verdict.

In [ ]:
# TODO: build a pandas Series from rf.feature_importances_ indexed by `honest`,
# sort it descending, and print each feature with its score.
# Then judge: does a high-cardinality feature top the list for the wrong reason?
# IMPORTANT: name the sorted Series exactly `imp` -- the chart in the next cell uses it.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
colours = ['tab:red' if n == 'submitted_hour' else 'tab:blue' for n in imp.index]
ax.barh(imp.index[::-1], imp.values[::-1], color=colours[::-1])
ax.set_xlabel('importance (mean decrease in impurity)')
ax.set_title('Random forest feature importance\n(red: high-cardinality, treat with suspicion)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

## 7. Beyond yes/no: the 3x3 confusion matrix

Real questions are often not binary. Bucket resolution speed into **Fast**, **On-time** and **Slow** relative to each service's SLA target, and predict that instead. The confusion matrix is now 3x3: the diagonal is correct, every off-diagonal cell is a specific confusion.

In [ ]:
from sklearn.metrics import confusion_matrix

spd = df[df['resolution_hours'].notna() & (df['resolution_hours'] >= 0)].copy()
ratio = spd['resolution_hours'] / spd['target_resolution_hours']
spd['speed'] = np.select([ratio <= 0.5, ratio <= 1.0], ['Fast', 'On-time'], default='Slow')
labels = ['Fast', 'On-time', 'Slow']

# TODO: split spd[honest] vs spd['speed'] (stratify=spd['speed'], random_state=42),
# fit a RandomForestClassifier, then build and print the 3x3 confusion_matrix with
# labels=labels. Look at which off-diagonal cells are emptiest and which is largest.
# IMPORTANT: name the matrix exactly `cm` -- the heatmap in the next cell uses it
# (along with `labels`, already defined above).


### See it: the 3x3 matrix as a heatmap

The adjacency pattern is hard to spot in a grid of numbers and impossible to miss in colour. Rows are normalised, so each row shows where that true class *went*.

In [ ]:
import matplotlib.pyplot as plt

norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i, j]}\n{norm[i, j]:.0%}', ha='center', va='center',
                fontsize=11, color='white' if norm[i, j] > 0.5 else 'black')
ax.set_xticks(range(3)); ax.set_xticklabels([f'predicted\n{l}' for l in labels])
ax.set_yticks(range(3)); ax.set_yticklabels([f'actual\n{l}' for l in labels])
ax.set_title('Where each true class goes (row-normalised)', pad=12)
# highlight the two far corners: the errors that barely happen
for (i, j) in [(0, 2), (2, 0)]:
    ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False,
                               edgecolor='red', linewidth=2.5))
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

Read it two ways, and note they do not say quite the same thing.

**By raw count**, the two far corners outlined in red are the smallest cells on the grid: the extreme confusions happen least often. **Row-normalised**, the picture sharpens for the Slow class in particular: only about 2% of genuinely slow requests are called Fast, while most of its errors land in the adjacent On-time bucket. That is the signature of an ordinal target, the model has learned the *ordering* even when it gets the exact bucket wrong, which a binary-only evaluation would never reveal.

Be careful with the Fast row, though. It holds far fewer requests than the other two, so its percentages swing on small counts and its Fast-to-Slow rate looks higher than the raw count suggests. This is worth pausing on: **row-normalising a rare class exaggerates its error rates**, and reading only the colours would have you overstate how often the model confuses fast with slow. Always check the counts behind the percentages.

## 8. Did complexity help?

Honest answer for this problem: barely. The ensembles earned their keep against an overfit tree, and the feature-importance and multiclass views told us *more* about the problem, but the headline accuracy ceiling held. Knowing when a fancier model is not worth it is a senior skill. Lab 3 changes the question entirely, from prediction to discovery, where there is no label at all.

## Your turn

1. Raise the shallow tree's `max_depth` from 4 towards the deep tree, one step at a time, watching train and test accuracy. At roughly what depth does the test score stop improving while the train score keeps rising?
2. Read the same feature importances from the XGBoost model (`xgb.feature_importances_`). Do the top features agree with the random forest, and does the ranking match what you learned about the data on Day 2?
3. In the 3x3 matrix, which single off-diagonal cell is largest? Phrase, in one business sentence, the mistake the model makes most often.

In [ ]:
# your turn
